In [1]:
!pip install emoji


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
#install nltk
!pip install nltk


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import emoji

In [4]:
def download_nltk_resources():
    resources = ['stopwords', 'punkt', 'wordnet', 'omw-1.4', 'punkt_tab']
    for resource in resources:
        try:
            nltk.download(resource, quiet=True)
        except Exception as e:
            print(f"Error downloading {resource}: {e}")

download_nltk_resources()

In [5]:
def clean_text(text):
    """
    Cleans the input text by removing URLs, mentions, hashtags, and special characters.
    """
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # Remove user mentions (@user)
    text = re.sub(r'@\w+', '', text)
    
    # Remove hashtags (keep the text, remove the symbol #)
    text = re.sub(r'#', '', text)
    
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [6]:
def handle_emojis(text):
    """
    Converts emojis into their textual descriptions.
    """
    # Demojize: converts emoji to its textual description (e.g., :smile:)
    return emoji.demojize(text, delimiters=(" ", " "))

In [7]:
def preprocess_tokens(text):
    """
    Tokenizes text, removes stopwords, and performs lemmatization.
    """
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    
    # Tokenize text
    tokens = word_tokenize(text)
    
    # Remove stopwords and lemmatize
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    
    return cleaned_tokens

In [8]:
def preprocess_tweet(tweet):
    """
    A master function to handle the entire preprocessing pipeline for a tweet.
    """
    if not isinstance(tweet, str):
        return ""
    
    # Handle emojis first to preserve their meaning
    tweet = handle_emojis(tweet)
    
    # Clean basic text noise
    tweet = clean_text(tweet)
    
    # Tokenize, remove stopwords, and lemmatize
    tokens = preprocess_tokens(tweet)
    
    # Join tokens back into a string
    return " ".join(tokens)

In [10]:
df = pd.read_csv('../combined_tweets_dataset/sample_v2/vibe_coding_combined_translated.csv')
df.head()

,conversation_id_str,created_at,favorite_count,full_text,id_str,image_url,in_reply_to_screen_name,lang,location,quote_count,reply_count,retweet_count,tweet_url,user_id_str,username,full_text_translated
0,1886192184808149383,2025-02-02 23:22:18+00:00,444,@karpathy me vibe coding and hope it can just ...,1886193454730137763,https://pbs.twimg.com/media/Gi0a82HaQAA97Q-.jpg,karpathy,en,NaN,0,5,10,https://x.com/Yuchenj_UW/status/18861934547301...,800854096219471872,Yuchenj_UW,@karpathy me vibe coding and hope it can just ...
1,1886196686709805486,2025-02-02 23:35:08+00:00,10,vibes capital meets vibe coding = greatest eco...,1886196686709805486,NaN,NaN,en,NaN,0,0,0,https://x.com/nikunj/status/1886196686709805486,30405893,nikunj,vibes capital meets vibe coding = greatest eco...
2,1886198988199600391,2025-02-02 23:44:17+00:00,3563,vibe coding https://t.co/1hHVMmunrw,1886198988199600391,https://pbs.twimg.com/media/Gi0f9fAWgAAgox6.jpg,NaN,en,NaN,18,35,191,https://x.com/IterIntellectus/status/188619898...,1655936149733654530,IterIntellectus,vibe coding https://t.co/1hHVMmunrw
3,1886203668619493715,2025-02-03 00:02:53+00:00,0,can attest. vibe coding is hella fun and borde...,1886203668619493715,NaN,NaN,en,NaN,0,0,0,https://x.com/kermankohli/status/1886203668619...,143624329,kermankohli,can attest. vibe coding is hella fun and borde...
4,1886204681766027415,2025-02-03 00:06:54+00:00,7,I have built a few app ideas in my spare time ...,1886204681766027415,NaN,NaN,en,NaN,0,0,0,https://x.com/luizfgparreira/status/1886204681...,2937425254,luizfgparreira,I have built a few app ideas in my spare time ...


In [11]:
# preprocess the tweets and store in a new column
df["translated_preprocessed_text"] = df["full_text_translated"].apply(preprocess_tweet)

In [12]:
df[["full_text_translated", "translated_preprocessed_text"]].head()

,full_text_translated,translated_preprocessed_text
0,@karpathy me vibe coding and hope it can just ...,vibe coding hope work
1,vibes capital meets vibe coding = greatest eco...,vibe capital meet vibe coding greatest economi...
2,vibe coding https://t.co/1hHVMmunrw,vibe coding
3,can attest. vibe coding is hella fun and borde...,attest vibe coding hella fun borderline like d...
4,I have built a few app ideas in my spare time ...,built app idea spare time one internal tool us...


In [13]:
# save the preprocessed dataset to a new CSV file
df.to_csv("../vibe_coding_translated_preprocessed.csv", index=False)